# Biểu đồ cho báo cáo khóa luận (hệ thống giám sát PPE)

Notebook vẽ lại mọi biểu đồ từ **file số liệu có trong git** (`outputs/results/**/*.json`, `results.csv`, `frames.csv`),
không cần trọng số, ảnh hay video. Ảnh lưu ở `outputs/figures/` (PNG 200 dpi để dán vào Word).

**Cách chạy:** mở bằng VS Code (extension Jupyter) hoặc `jupyter notebook`, chọn kernel `.venv`, **Run All**. Cần
`matplotlib` (có trong `requirements.txt`) và `ipykernel` / `notebook` để chạy notebook. Ô nào thiếu file số liệu sẽ in
"Bỏ qua: …" thay vì báo lỗi.

**Lưu ý khi đưa vào báo cáo:**
- Số trên **val** (chọn bản), chưa phải kết quả cuối. Kết quả cuối lấy từ **test, chạy một lần** cho bản được chọn.
- Val, test đã sửa là **nhãn máy, có kiểm định mẫu của người** (`docs/experiments/2026-11-16_c5_detector_plan.md`).
- Holdout (RQ2), ablation (RQ3), benchmark (RQ4) hiện đo với **hệ thống bản C4**; khi chốt bản cuối phải chạy lại.
- Thêm bản mới (vd C7): sửa danh sách `RUNS` ở ô cấu hình bên dưới.

In [ ]:
import csv, json
from collections import defaultdict
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline.yaml").exists() and ROOT != ROOT.parent:   # chạy được từ scripts/ hoặc thư mục gốc
    ROOT = ROOT.parent
assert (ROOT / "configs" / "pipeline.yaml").exists(), "Mở notebook trong repo PPE_Project"
OUT = ROOT / "outputs" / "figures"
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 10, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

PPE = ["helmet", "vest", "gloves", "goggles", "boots"]
CLASSES = ["person"] + PPE
VI = {"person": "Người", "helmet": "Mũ", "vest": "Áo", "gloves": "Găng", "goggles": "Kính", "boots": "Ủng"}
COLORS = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860", "#DA8BC3"]


def load_json(rel):
    p = ROOT / rel
    if not p.exists():
        print("Bỏ qua: thiếu", rel)
        return None
    return json.loads(p.read_text(encoding="utf-8"))


def save(fig, name):
    fig.tight_layout()
    fig.savefig(OUT / f"{name}.png", bbox_inches="tight")
    print("Đã lưu", (OUT / f"{name}.png").relative_to(ROOT))
    plt.show()

print("ROOT =", ROOT)

## Cấu hình: các phiên bản detector để so sánh (RQ1)

Mỗi bản: tên hiển thị, file so sánh của `04j_c5_eval.py` (`outputs/results/c5_eval/<file>.json`) và khóa của bản đó trong
file. Mọi số chấm trên **val đã sửa**. Thêm C7 khi có: `("C7-m", "c6_vs_c7.json", "c7_m")`.

In [ ]:
RUNS = [
    ("C4",         "c4_vs_c5_s960.json",     "c4"),
    ("C5-s960",    "c4_vs_c5_s960.json",     "c5_s960"),
    ("C5-s1280",   "c5_960_vs_1280.json",    "c5_s1280"),
    ("C6-scratch", "c5_vs_c6_scratch.json",  "c6_scratch"),
    ("C6-ft",      "c5_vs_c6_ft.json",       "c6_ft"),
]
SUBSET_OF = {"person": "ppe4", "helmet": "ppe4", "vest": "ppe4", "gloves": "ppe4", "goggles": "goggles_all", "boots": "new"}
TARGET = 0.8

table = {}   # (bản, lớp) -> (map50, ci_lo, ci_hi, gt)
for name, f, key in RUNS:
    d = load_json(f"outputs/results/c5_eval/{f}")
    if d is None:
        continue
    for c in CLASSES:
        row = d["subsets"].get(SUBSET_OF[c], {}).get("classes", {}).get(c)
        if row and key in row:
            table[(name, c)] = (row[key]["map50"], *row[key]["ci50"], row["gt"])
names = [n for n, _, _ in RUNS if any((n, c) in table for c in CLASSES)]
print(f"{'Lớp':8s}" + "".join(f"{n:>12s}" for n in names))
for c in CLASSES:
    print(f"{VI[c]:8s}" + "".join(f"{table[(n, c)][0]:12.3f}" if (n, c) in table else f"{'-':>12s}" for n in names))

## Hình 1 (RQ1). mAP50 từng lớp qua các phiên bản, khoảng tin cậy 95% (bootstrap theo cảnh), mốc 0,8

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.6))
w = 0.8 / max(len(names), 1)
x = np.arange(len(CLASSES))
for i, n in enumerate(names):
    v = [table.get((n, c), (np.nan, np.nan, np.nan, 0)) for c in CLASSES]
    m = np.array([t[0] for t in v]); lo = np.array([t[1] for t in v]); hi = np.array([t[2] for t in v])
    ax.bar(x + (i - (len(names) - 1) / 2) * w, m, w, yerr=[m - lo, hi - m], capsize=2, label=n, color=COLORS[i % len(COLORS)])
ax.axhline(TARGET, color="red", ls="--", lw=1.2, label=f"Mục tiêu {TARGET}")
gt = [table.get((names[-1], c), (0, 0, 0, 0))[3] for c in CLASSES]
ax.set_xticks(x, [f"{VI[c]}\n({g} hộp)" for c, g in zip(CLASSES, gt)])
ax.set_ylim(0, 1); ax.set_ylabel("mAP@0.5 (val đã sửa)")
ax.set_title("mAP@0.5 từng lớp qua các phiên bản detector")
ax.legend(ncol=4, fontsize=8, loc="lower left")
save(fig, "fig01_map50_versions")

## Hình 2 (RQ1). Đường cong học (mAP trên val_monitor theo epoch)

In [ ]:
TRAIN_RUNS = [("C4", "yolo11s_ppe6_c4"), ("C5-s960", "yolo11s_ppe6_c5_s960"), ("C5-s1280", "yolo11s_ppe6_c5_s1280"),
              ("C6-scratch", "yolo11s_ppe6_c6_scratch"), ("C6-ft", "yolo11s_ppe6_c6_ft")]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for i, (n, run) in enumerate(TRAIN_RUNS):
    p = ROOT / "outputs" / "train" / run / "results.csv"
    if not p.exists():
        print("Bỏ qua: thiếu", p.relative_to(ROOT)); continue
    rows = list(csv.DictReader(open(p, encoding="utf-8")))
    ep = [int(float(r["epoch"])) for r in rows]
    axes[0].plot(ep, [float(r["metrics/mAP50(B)"]) for r in rows], label=n, color=COLORS[i])
    axes[1].plot(ep, [float(r["train/box_loss"]) for r in rows], label=n, color=COLORS[i])
axes[0].set_title("mAP@0.5 trên val_monitor (nhãn cũ, chỉ để theo dõi)"); axes[0].set_xlabel("epoch")
axes[1].set_title("box loss khi train"); axes[1].set_xlabel("epoch")
axes[0].legend(fontsize=8)
save(fig, "fig02_learning_curves")

## Hình 3 (RQ1). Recall theo cỡ hộp (cạnh ngắn ở đầu vào 960 px) — vì sao găng thấp

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
src = [("Áo", "c6ft_val_ppe4", "vest"), ("Găng", "c6ft_val_ppe4", "gloves"), ("Ủng", "c6ft_val_boots", "boots")]
bins = ["<16", "<32", "<64", ">=64"]
lab = {"<16": "< 16", "<32": "16–32", "<64": "32–64", ">=64": "≥ 64"}
for i, (vi, run, c) in enumerate(src):
    d = load_json(f"outputs/results/error_analysis/{run}/summary.json")
    if d is None:
        continue
    b = d["classes"][c]["by_size"]
    r = [b.get(k, {}).get("recall", np.nan) for k in bins]
    n = [b.get(k, {}).get("total", 0) for k in bins]
    xs = np.arange(len(bins)) + (i - 1) * 0.27
    ax.bar(xs, r, 0.27, label=vi, color=COLORS[i + 1])
    for xx, rr, nn in zip(xs, r, n):
        ax.text(xx, (rr if rr == rr else 0) + 0.02, str(nn), ha="center", fontsize=7)
ax.set_xticks(range(len(bins)), [lab[k] + " px" for k in bins]); ax.set_ylim(0, 1.1)
ax.set_ylabel("Recall tối đa (C6-ft, val đã sửa)"); ax.set_title("Recall theo cỡ hộp (số trên cột: số hộp)")
ax.legend()
save(fig, "fig03_recall_by_size")

## Hình 4 (RQ1). Sửa nhãn val làm thay đổi phép đo: C4 trên val gốc và val đã sửa

In [ ]:
old = load_json("outputs/results/ppe6_eval/c4_ppe4val/ultralytics.json")
oldnew = load_json("outputs/results/ppe6_eval/c4_valnew.json")
if old and oldnew:
    v_old = {"person": old["ap50"]["0"], "helmet": old["ap50"]["1"], "vest": old["ap50"]["2"], "gloves": old["ap50"]["3"],
             "goggles": oldnew["classes"]["goggles"]["map50"], "boots": oldnew["classes"]["boots"]["map50"]}
    v_new = {c: table[("C4", c)][0] for c in CLASSES if ("C4", c) in table}
    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(CLASSES))
    ax.bar(x - 0.2, [v_old[c] for c in CLASSES], 0.4, label="Val gốc", color="#999999")
    ax.bar(x + 0.2, [v_new.get(c, np.nan) for c in CLASSES], 0.4, label="Val đã sửa", color=COLORS[0])
    ax.set_xticks(x, [VI[c] for c in CLASSES]); ax.set_ylim(0, 1); ax.set_ylabel("mAP@0.5 của C4")
    ax.set_title("Cùng mô hình C4: val gốc so với val đã sửa nhãn"); ax.legend()
    save(fig, "fig04_val_relabel_effect")

## Hình 5 (RQ1, RQ4). Tốc độ detector trên laptop (RTX 3050 4 GB), mốc 20 khung/giây cho cả hệ thống

In [ ]:
rows = []
d = load_json("outputs/results/error_analysis/detector_speed.json")
if d:
    for r in d["runs"]:
        if "yolo11s_ppe6" in r["weights"] or "yolo11m" in r["weights"]:
            m = "s (C4)" if "yolo11s_ppe6" in r["weights"] else "m"
            rows.append((f"{m} @{r['imgsz']} FP32", r["mean_ms"], r["p95_ms"]))
d = load_json("outputs/results/error_analysis/detector_speed_fp16.json")
if d:
    for r in d["runs"]:
        if r.get("half"):
            m = "s (C6)" if "yolo11s" in r["weights"] else "m"
            rows.append((f"{m} @{r['imgsz']} FP16", r["mean_ms"], r["p95_ms"]))
if rows:
    rest = 21.0   # ms/khung của phần còn lại của pipeline (benchmark bản cuối 37,7 ms trừ detector 17 ms), ước lượng
    fig, ax = plt.subplots(figsize=(9, 4))
    y = np.arange(len(rows))
    ax.barh(y, [r[1] for r in rows], color=COLORS[0], label="detector, ms/khung (TB)")
    ax.barh(y, [rest] * len(rows), left=[r[1] for r in rows], color="#cccccc", label="phần còn lại pipeline (ước lượng)")
    ax.axvline(50, color="red", ls="--", label="50 ms = 20 khung/giây")
    ax.set_yticks(y, [r[0] for r in rows]); ax.set_xlabel("ms / khung")
    ax.set_title("Tốc độ detector theo cỡ mô hình, cỡ ảnh, độ chính xác số")
    ax.legend(fontsize=8)
    save(fig, "fig05_detector_speed")

## Hình 6 (RQ2). Ma trận 3×3 trạng thái PPE trên holdout

Hàng: nhãn (Có / Thiếu / Chưa xác định); cột: hệ thống. Holdout 8 video, **nhãn máy, chưa người kiểm định**, hệ thống
bản cuối khi đó (C4). Khi chốt bản mới phải chạy lại holdout.

In [ ]:
h = load_json("outputs/results/close_eval/final/holdout/results.json")
STATES = ["Có", "Thiếu", "Chưa XĐ"]


def matrix(s):
    co = [s["co_dung"], s["bao_thieu_oan"], s["co_thanh_chua_xd"]]
    th = [s["bao_co_sai"], s["thieu_dung"], s["thieu_thanh_chua_xd"]]
    cx_co, cx_th = s["co_tren_nhan_chua_xd"], s["thieu_tren_nhan_chua_xd"]
    unm = s["khong_ghep"]
    cx = [cx_co, cx_th, max(s["nhan_chua_xac_dinh"] - cx_co - cx_th - unm, 0)]
    return np.array([co, th, cx], dtype=float)


mats = {}
if h:
    tot = h["totals"]["final"]
    fig, axes = plt.subplots(1, len(PPE), figsize=(16, 3.6))
    for ax, c in zip(axes, PPE):
        m = matrix(tot[c]); mats[c] = m
        nrm = m / np.maximum(m.sum(1, keepdims=True), 1)
        ax.imshow(nrm, cmap="Blues", vmin=0, vmax=1)
        for i in range(3):
            for j in range(3):
                ax.text(j, i, f"{int(m[i, j])}\n{nrm[i, j]:.0%}", ha="center", va="center", fontsize=7,
                        color="white" if nrm[i, j] > 0.6 else "black")
        ax.set_xticks(range(3), STATES, fontsize=8); ax.set_yticks(range(3), STATES, fontsize=8)
        ax.set_title(VI[c]); ax.set_xlabel("Hệ thống"); ax.grid(False)
    axes[0].set_ylabel("Nhãn")
    fig.suptitle("Ma trận trạng thái trên holdout (số khung-người; % theo hàng)")
    save(fig, "fig06_state_confusion_holdout")

## Hình 7 (RQ2). P / R / F1 từng trạng thái, macro-F1 và Coverage

In [ ]:
if mats:
    rows = {}
    for c, m in mats.items():
        f1s = []
        for k in range(3):
            tp = m[k, k]; p = tp / max(m[:, k].sum(), 1); r = tp / max(m[k, :].sum(), 1)
            f1s.append(2 * p * r / max(p + r, 1e-9))
        cov = m[:, :2].sum() / max(m.sum(), 1)   # Coverage = (N_có + N_thiếu) / N_tổng (spec 7.3)
        rows[c] = (*f1s, float(np.mean(f1s)), cov)
    fig, ax = plt.subplots(figsize=(10, 4))
    lab = ["F1 Có", "F1 Thiếu", "F1 Chưa XĐ", "macro-F1", "Coverage"]
    x = np.arange(len(PPE)); w = 0.16
    for i, l in enumerate(lab):
        ax.bar(x + (i - 2) * w, [rows[c][i] for c in PPE], w, label=l, color=COLORS[i])
    ax.set_xticks(x, [VI[c] for c in PPE]); ax.set_ylim(0, 1.05)
    ax.set_title("Trạng thái PPE trên holdout: F1 từng trạng thái, macro-F1, Coverage"); ax.legend(ncol=5, fontsize=8)
    save(fig, "fig07_state_f1_coverage")
    for c in PPE:
        print(f"{VI[c]:6s} " + "  ".join(f"{l} {v:.3f}" for l, v in zip(lab, rows[c])))

## Hình 8 (RQ3). Ablation A → E: số cảnh báo và Event F1

In [ ]:
d = load_json("outputs/results/ablation/ablation_results.json")
if d:
    s = d["summary"]
    st = [r["stage"] for r in s]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].bar(st, [r["tp"] for r in s], label="đúng", color=COLORS[2])
    axes[0].bar(st, [r["fp_false"] for r in s], bottom=[r["tp"] for r in s], label="sai", color=COLORS[3])
    axes[0].bar(st, [r["fp_dup"] for r in s], bottom=[r["tp"] + r["fp_false"] for r in s], label="trùng", color="#bbbbbb")
    axes[0].set_yscale("log"); axes[0].set_ylabel("số cảnh báo (thang log)"); axes[0].legend(); axes[0].set_title("Cảnh báo theo cấu hình")
    axes[1].plot(st, [r["precision"] for r in s], "o-", label="Precision")
    axes[1].plot(st, [r["recall"] for r in s], "s-", label="Recall")
    axes[1].plot(st, [r["f1"] for r in s], "^-", label="Event F1")
    axes[1].set_ylim(0, 1.05); axes[1].legend(); axes[1].set_title("Precision / Recall / F1 sự kiện")
    fig.suptitle("Ablation: A YOLO thuần → B vùng cơ thể → C ByteTrack → D cửa sổ thời gian → E vòng đời sự kiện")
    save(fig, "fig08_ablation")
    for r in s:
        print(r["name"], "| F1", r["f1"], "| độ trễ TB", r["mean_latency_s"], "s")

## Hình 9 (RQ3). Dòng thời gian trạng thái PPE và sự kiện trên một video

In [ ]:
VIDEO = "video4"
p = ROOT / "outputs" / "results" / "video" / VIDEO / "frames.csv"
if not p.exists():
    print("Bỏ qua: thiếu", p.relative_to(ROOT))
else:
    rows = list(csv.DictReader(open(p, encoding="utf-8")))
    tracks = sorted({int(r["track_id"]) for r in rows})[:3]
    code_of = {"co": 2, "thieu": 0, "chua_xac_dinh": 1}
    fig, axes = plt.subplots(len(tracks), 1, figsize=(11, 1.9 * len(tracks)), sharex=True, squeeze=False)
    cmap = matplotlib.colors.ListedColormap(["#d62728", "#dddddd", "#2ca02c"])
    for ax, tid in zip(axes[:, 0], tracks):
        rs = [r for r in rows if int(r["track_id"]) == tid]
        t = np.array([float(r["t"]) for r in rs])
        img = np.array([[code_of.get(r[c], 1) for r in rs] for c in PPE])
        ax.imshow(img, aspect="auto", cmap=cmap, vmin=0, vmax=2, extent=[t.min(), t.max(), len(PPE) - 0.5, -0.5], interpolation="nearest")
        ax.set_yticks(range(len(PPE)), [VI[c] for c in PPE]); ax.set_ylabel(f"Người #{tid}"); ax.grid(False)
    ev = ROOT / "outputs" / "results" / "video" / VIDEO / "events.jsonl"
    if ev.exists():
        for line in ev.read_text(encoding="utf-8").splitlines():
            e = json.loads(line)
            if e.get("change") == "created" and e["track_id"] in tracks:
                ax = axes[tracks.index(e["track_id"]), 0]
                ax.axvline(e["confirmed_at"], color="black", lw=1.5)
                ax.text(e["confirmed_at"], -0.6, " cảnh báo: " + ",".join(VI[m] for m in e["missing_items"]), fontsize=7, va="bottom")
    axes[-1, 0].set_xlabel("giây")
    fig.suptitle(f"{VIDEO}: trạng thái theo thời gian (xanh: Có, đỏ: Thiếu, xám: Chưa xác định); vạch đen: lúc xác nhận cảnh báo")
    save(fig, "fig09_timeline_" + VIDEO)

## Hình 10 (RQ4). Thời gian từng bước của pipeline (ms/khung, trung bình 4 video phát triển)

In [ ]:
b = load_json("outputs/results/close_eval/final/benchmark_clean/benchmark.json")
if b:
    by = defaultdict(lambda: defaultdict(list))
    lat = defaultdict(list)
    for r in b["runs"]:
        for k, v in r["step_ms_mean"].items():
            by[r["config"]][k].append(v)
        lat[r["config"]].append((r["latency_ms"]["p50"], r["latency_ms"]["p95"], r["process_fps"]))
    steps = ["detector", "pose", "parts", "tracker", "association", "compliance", "temporal_events"]
    vi = {"detector": "phát hiện", "pose": "pose", "parts": "phân loại bộ phận", "tracker": "theo dõi",
          "association": "gán PPE", "compliance": "trạng thái", "temporal_events": "thời gian + sự kiện"}
    cfgs = list(by)
    fig, ax = plt.subplots(figsize=(9, 3.2))
    left = np.zeros(len(cfgs))
    for i, s in enumerate(steps):
        v = np.array([np.mean(by[c][s]) for c in cfgs])
        ax.barh(cfgs, v, left=left, label=vi[s], color=COLORS[i % len(COLORS)])
        left += v
    ax.set_xlabel("ms / khung"); ax.legend(ncol=4, fontsize=8, loc="lower right")
    ax.set_title("Thời gian từng bước (benchmark máy rảnh; hệ thống bản C4 — đo lại khi chốt bản cuối)")
    save(fig, "fig10_step_time")
    for c in cfgs:
        a = np.array(lat[c]); print(c, f"p50 {a[:, 0].mean():.1f} ms, p95 {a[:, 1].mean():.1f} ms, FPS xử lý {a[:, 2].mean():.1f}")

## Hình 11 (dữ liệu). Số hộp mỗi lớp theo nguồn trong tập train C5

In [ ]:
d = load_json("outputs/results/c5_eval/data_stats.json")
if d:
    bx = d["c5"]["boxes"]
    srcs = sorted({k.split("|")[1] for k in bx})
    vi_src = {"sh17": "SH17", "chv": "CHV", "cppe_cu": "Construction-PPE cũ", "cppe_moi": "Construction-PPE mới",
              "cat_nguoi": "ảnh cắt người", "thu_nho": "ảnh thu nhỏ"}
    fig, ax = plt.subplots(figsize=(10, 4))
    bottom = np.zeros(len(CLASSES))
    for i, s in enumerate(srcs):
        v = np.array([bx.get(f"{c}|{s}", 0) for c in CLASSES])
        ax.bar([VI[c] for c in CLASSES], v, bottom=bottom, label=vi_src.get(s, s), color=COLORS[i % len(COLORS)])
        bottom += v
    ax.set_ylabel("số hộp"); ax.set_title("Train C5: số hộp mỗi lớp theo nguồn (C6, C7 thêm ủng từ Roboflow)")
    ax.legend(fontsize=8)
    save(fig, "fig11_train_sources")

## Hình 12 (dữ liệu). Tỉ lệ hộp nhỏ theo cỡ ảnh đầu vào 960 và 1280

In [ ]:
if d:
    bins = ["< 16", "16–32", "32–64", "≥ 64"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
    for ax, z in zip(axes, ["960", "1280"]):
        sh = d["c5"]["size_share"][z]
        left = np.zeros(len(CLASSES))
        for i, b in enumerate(bins):
            v = np.array([sh[c][i] for c in CLASSES])
            ax.barh([VI[c] for c in CLASSES], v, left=left, label=b + " px", color=plt.cm.viridis(i / 3))
            left += v
        ax.set_title(f"imgsz {z}"); ax.set_xlim(0, 1); ax.set_xlabel("tỉ lệ hộp")
    axes[1].legend(fontsize=8, loc="lower right")
    fig.suptitle("Cỡ hộp (cạnh ngắn) sau khi đưa ảnh về imgsz, train C5")
    save(fig, "fig12_box_size_share")

## Biểu đồ không vẽ ở đây

- Đường cong P–R, ma trận nhầm lẫn của detector: Ultralytics sinh sẵn trong `outputs/train/<run>/` (`BoxPR_curve.png`,
  `confusion_matrix_normalized.png`), chỉ có trên máy (không có trong git).
- Sơ đồ kiến trúc, sơ đồ quy trình dữ liệu: vẽ tay (draw.io, PowerPoint).
- Ảnh chụp dashboard, app desktop, khung camera: chụp khi chạy hệ thống.